# ClinEval — HPO Extraction Demo (Module A)

**ClinEval is an evaluator, not a model-builder.** It scores a system's clinical-text-to-HPO-term extractions against a gold standard and produces both quality metrics and a regulatory-evidence mapping. Nothing here trains or fine-tunes a model — the "system under test" is a pluggable extractor (a cached replay of prior predictions by default, or a live call to a local OpenAI-compatible endpoint such as LM Studio).

This notebook mirrors exactly what `clineval run --dataset synthetic` does on the command line, but inline so you can inspect every intermediate step:

1. Load the 10-record synthetic gold dataset and normalize HPO IDs.
2. Replay cached model predictions (no network calls — fully offline and reproducible) and normalize them too.
3. Load the HPO ontology and align both gold and system IDs (alt_id resolution, obsolete flagging).
4. Evaluate: Tier 1 exact match, Tier 2 semantic similarity (best-match Lin + IC-weighted + BMA), Tier 3 clinical error taxonomy.
5. Render the full Markdown report, including the exact-vs-semantic F1 gap and the regulatory-evidence mapping.

The timestamp below is fixed (not "now") so the notebook's output is deterministic and reproducible on any machine.

In [ ]:
from clineval.core.dataset import JSONLDatasetLoader
from clineval.core.evaluator import evaluate
from clineval.core.metric import EvalContext
from clineval.core.ontology.hpo import Ontology
from clineval.core.report import render_report
from clineval.tasks.hpo_extraction import adapters
from clineval.tasks.hpo_extraction.extractor import CachedExtractor
import clineval.tasks.hpo_extraction  # noqa: F401  (registers metrics)

records = JSONLDatasetLoader("data/synthetic_mini.jsonl").load()
for r in records:
    adapters.normalize_record(r)

extractor = CachedExtractor("data/cached_predictions.jsonl")
for r in records:
    r.system_output = extractor.extract(r)
    adapters.normalize_record(r)

ontology = Ontology()
records, alignment = adapters.align_records(records, ontology)
result = evaluate(
    "hpo_extraction", records, EvalContext(ontology=ontology),
    dataset="synthetic", model=f"cached:{extractor.model}",
    timestamp="2026-07-09T00:00:00+00:00", alignment=alignment,
)
print("exact F1:", result.metric("tier1_exact").aggregate["f1"])
print("semantic F1:", result.metric("tier2_semantic").aggregate["sem_f1"])

## Full report

The same Markdown report the CLI writes to `reports/report.md` — including per-tier precision/recall/F1, the exact-vs-semantic gap, the Tier 3 error taxonomy and clinical-significance flags, the ontology-alignment summary, and the regulatory-evidence mapping table (EU AI Act / IVDR / ISO 15189:2022).

In [ ]:
from IPython.display import Markdown
Markdown(render_report(result))